<a href="https://colab.research.google.com/github/Rahmatul-Rovi/Data-Mining-Lab-Tasks/blob/main/Association_Rule_Data_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

rahmatulrovi386_retail_transaction_path = kagglehub.dataset_download('rahmatulrovi386/retail-transaction')
prasad22_retail_transactions_dataset_path = kagglehub.dataset_download('prasad22/retail-transactions-dataset')

print('Data source import complete.')


In [ ]:
import warnings; warnings.filterwarnings("ignore")
!pip install -q mlxtend kagglehub

import pandas as pd, ast, os, kagglehub
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

SAMPLE = 100000
THRESHOLDS = [0.01, 0.005, 0.002, 0.001]
KAGGLE = "https://www.kaggle.com/datasets/prasad22/retail-transactions-dataset"

In [ ]:
# 1. Load
path = kagglehub.dataset_download("prasad22/retail-transactions-dataset")
csv = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv), usecols=["Product"])
N_TOTAL = len(df)
if SAMPLE: df = df.sample(SAMPLE, random_state=42)
N_USED = len(df)
transactions = [list(set(ast.literal_eval(x))) for x in df["Product"]]

lens = [len(t) for t in transactions]
U = len({i for t in transactions for i in t})
AVG, MN, MX = round(sum(lens)/len(lens), 2), min(lens), max(lens)

In [ ]:
# 2. Boolean conversion
te = TransactionEncoder()
basket = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
SHAPE = basket.shape

In [ ]:
# 3. Frequent itemsets
def rules_from(fi):
    try:
        return association_rules(fi, metric="confidence", min_threshold=0.0)
    except TypeError:
        return association_rules(fi, num_itemsets=len(basket), metric="confidence", min_threshold=0.0)

res = {}
for s in THRESHOLDS:
    fi = fpgrowth(basket, min_support=s, use_colnames=True, max_len=3)
    fi["size"] = fi["itemsets"].apply(len)
    res[s] = fi
    print(f"min_support={s}: total={len(fi)}", fi["size"].value_counts().sort_index().to_dict())

low = min(THRESHOLDS)
fi_low = res[low]
top_sets = fi_low[fi_low["size"] >= 2].sort_values("support", ascending=False).head(5)

In [ ]:
# 4. Rules
rules = rules_from(fi_low)
rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))
rules = rules.sort_values("lift", ascending=False)
top_rules = rules.head(5)
MAXLIFT, MEANLIFT = rules["lift"].max(), rules["lift"].mean()
print("\nTotal rules:", len(rules), "| max lift:", round(MAXLIFT,3), "| mean lift:", round(MEANLIFT,3))
print(top_rules[["antecedents","consequents","support","confidence","lift"]].to_string(index=False))
print("\nTop itemsets (size>=2):\n", top_sets[["itemsets","support"]].to_string(index=False))